## Recommendation System Assignment
#### Recommendation System

Data Description:

Unique ID of each anime.
Anime title.
Anime broadcast type, such as TV, OVA, etc.
anime genre.
The number of episodes of each anime.
The average rating for each anime compared to the number of users who gave ratings.


Number of community members for each anime.
Objective:
The objective of this assignment is to implement a recommendation system using cosine similarity on an anime dataset.
Dataset:
Use the Anime Dataset which contains information about various anime, including their titles, genres,No.of episodes and user ratings etc.

Tasks:

Data Preprocessing:

Load the dataset into a suitable data structure (e.g., pandas DataFrame).
Handle missing values, if any.
Explore the dataset to understand its structure and attributes.

Feature Extraction:

Decide on the features that will be used for computing similarity (e.g., genres, user ratings).
Convert categorical features into numerical representations if necessary.
Normalize numerical features if required.

Recommendation System:

Design a function to recommend anime based on cosine similarity.
Given a target anime, recommend a list of similar anime based on cosine similarity scores.
Experiment with different threshold values for similarity scores to adjust the recommendation list size.

Evaluation:

Split the dataset into training and testing sets.
Evaluate the recommendation system using appropriate metrics such as precision, recall, and F1-score.
Analyze the performance of the recommendation system and identify areas of improvement.

Interview Questions:
1. Can you explain the difference between user-based and item-based collaborative filtering?
2. What is collaborative filtering, and how does it work?

In [1]:
# Step 1: Import Required Libraries
# Import necessary libraries for data handling, visualization, and building the recommendation system.
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

In [3]:
# Step 2: Load and Explore the Dataset
# Load the "anime.csv" dataset.
dataset_path = 'anime.csv'
data = pd.read_csv(dataset_path)

# Display the first few rows of the dataset
data.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama&#039;,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [4]:
#summarize the dataset
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12294 non-null  int64  
 1   name      12294 non-null  object 
 2   genre     12232 non-null  object 
 3   type      12269 non-null  object 
 4   episodes  12294 non-null  object 
 5   rating    12064 non-null  float64
 6   members   12294 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 672.5+ KB


In [5]:
# Check for missing values in the dataset
data.isnull().sum()

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64

In [6]:
data.describe()

,anime_id,rating,members
count,12294.000000,12064.000000,1.229400e+04
mean,14058.221653,6.473902,1.807134e+04
std,11455.294701,1.026746,5.482068e+04
min,1.000000,1.670000,5.000000e+00
25%,3484.250000,5.880000,2.250000e+02
50%,10260.500000,6.570000,1.550000e+03
75%,24794.500000,7.180000,9.437000e+03
max,34527.000000,10.000000,1.013917e+06


In [9]:
#impute missing values using mean for numerical columns
data['rating'] = data['rating'].fillna(data['rating'].mean())
data['members'] = data['members'].fillna(data['members'].mean())

# impute missing values using mode for categorical columns
data['type'] = data['type'].fillna(data['type'].mode()[0])
data['episodes'] = data['episodes'].fillna(data['episodes'].mode()[0])
data['genre'] = data['genre'].fillna(data['genre'].mode()[0])


In [10]:
data.isnull().sum()

anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64

In [12]:
# Step 4: Feature Extraction
# Select features for the recommendation system (e.g., genres, ratings, etc.)
# Encoding the 'genre' column using OneHotEncoder
encoder = OneHotEncoder()
genre_encoded = encoder.fit_transform(data[['genre']]).toarray()

In [13]:
genre_encoded

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [15]:
# Normalize numerical features (e.g., ratings)
scaler = StandardScaler()
data['normalized_rating'] = scaler.fit_transform(data[['rating']])

In [16]:
# Combine features into a single matrix for similarity computation
features = np.hstack([genre_encoded, data[['normalized_rating']]])

In [17]:
data['normalized_rating'].head()

0    2.847535
1    2.739380
2    2.729547
3    2.650889
4    2.641057
Name: normalized_rating, dtype: float64

In [26]:
def recommend_anime(anime_title, data, features, top_n=5):
    # Find the index of the target anime
    if anime_title not in data['name'].values:
        print(f"Anime '{anime_title}' not found in the dataset.")
        return []

    anime_index = data[data['name'] == anime_title].index[0]

    # Compute cosine similarity between the target anime and all others
    similarity_scores = cosine_similarity(features[anime_index].reshape(1, -1), features).flatten()

    # Sort scores and fetch indices of top matches (excluding the anime itself)
    similar_indices = similarity_scores.argsort()[::-1][1:top_n + 1] # [::-1] means sort in descending order and [1:] excludes the first element (the anime itself) from the top matches and [1:top_n + 1] to get top n matches excluding the anime itself from the top matches

    # Return titles of the most similar anime
    recommended_anime = data.iloc[similar_indices][['name', 'rating', 'genre']]
    return recommended_anime

In [31]:
# Example usage of the recommendation function
print("\nRecommendations for 'Naruto':")
recommendations = recommend_anime('One Piece', data, features)
print(recommendations)


Recommendations for 'Naruto':
                                                                                         name  \
231                          One Piece: Episode of Merry - Mou Hitori no Nakama no Monogatari   
241                      One Piece: Episode of Nami - Koukaishi no Namida to Nakama no Kizuna   
896    One Piece: Episode of Sabo - 3 Kyoudai no Kizuna Kiseki no Saikai to Uketsugareru Ishi   
10464                                                 Taka no Tsume 8: Yoshida-kun no X-Files   
10400                                                             Spoon-hime no Swing Kitchen   

       rating                                                            genre  
231      8.29  Action, Adventure, Comedy, Drama, Fantasy, Shounen, Super Power  
241      8.27  Action, Adventure, Comedy, Drama, Fantasy, Shounen, Super Power  
896      7.78  Action, Adventure, Comedy, Drama, Fantasy, Shounen, Super Power  
10464   10.00                                                 

In [28]:
# Step 6: Evaluation
# Split the dataset into training and testing sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)


In [29]:
# For simplicity, assume precision and recall are calculated based on genre overlap
def evaluate_recommendation_system(test_data, features, data):
    precision_scores = []
    recall_scores = []

    for anime_title in test_data['name']:
        recommendations = recommend_anime(anime_title, data, features, top_n=5)
        if len(recommendations) == 0:
            continue

        # Calculate precision and recall (assuming genres are true labels)
        target_genres = set(data[data['name'] == anime_title]['genre'].values[0].split(','))
        recommended_genres = set(','.join(recommendations['genre']).split(','))

        tp = len(target_genres & recommended_genres)
        fp = len(recommended_genres - target_genres)
        fn = len(target_genres - recommended_genres)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0

        precision_scores.append(precision)
        recall_scores.append(recall)

    avg_precision = np.mean(precision_scores)
    avg_recall = np.mean(recall_scores)
    f1 = 2 * (avg_precision * avg_recall) / (avg_precision + avg_recall) if (avg_precision + avg_recall) > 0 else 0

    return avg_precision, avg_recall, f1


In [30]:
# Calculate evaluation metrics
precision, recall, f1 = evaluate_recommendation_system(test_data, features, data)
print("\nEvaluation Metrics:")
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1 Score: {f1:.2f}")


Evaluation Metrics:
Precision: 0.71
Recall: 0.87
F1 Score: 0.78


## Interview Questions:
### Step 7: Interview Questions
#### 1. Can you explain the difference between user-based and item-based collaborative filtering?
####    User-based collaborative filtering recommends items based on the similarity between users' preferences.
####    Item-based collaborative filtering recommends items based on the similarity between item attributes or ratings by users.

#### 2. What is collaborative filtering, and how does it work?
####    Collaborative filtering predicts user preferences by finding patterns in user-item interactions (e.g., ratings) and leveraging the preferences of similar users or items.
